### 統計的裁定

勝率の高そうな状況を作り出して長期的な勝利を狙う戦術

#### 定義

時刻$t$におけるポートフォリオの価値: $V(t)$

以下を満たすようなポートフォリオを構築可能 $\Rightarrow$ 市場に統計的裁定機会が存在

1. $V(0) = 0$

2. $\lim_{T \rightarrow \infty} {\mathbb E} [V(t)] = 0$

3. $\lim_{T \rightarrow \infty} P(V(T) < 0) = 0$

4. 任意の$T$に対して$P(V(T) < 0) > 0 \Rightarrow \lim_{T \rightarrow \infty} \frac {Var(V(T))} {T} = 0$

裁定機会の拡張となっているのは4.のおかげ

### 共和分法

定常性(stationarity)をポートフォリオに持たせて統計的裁定を狙う手法

定常性のあるポートフォリオの価値 ${\mathbb E} [V(T)] = \mu_P$

$V(0) < \mu_P$ならばロングを狙う (条件が逆ならばショート)

#### 定常性の定義

確率過程$X(t)$が以下の性質を満足 $\Rightarrow$ (弱)定常

1. ${\mathbb E} [X(t)] = \mu$

2. $Cov (X(t), X(t-s)) = \gamma (s)$

$\mu$: const.

$\gamma (s)$: 時差$s$のみに依存する関数 $\gamma (0) = Var(X(t))$

要は"データを生成する分布が一定"ということ

#### 定常性と統計的裁定

ポートフォリオが定常 $\Rightarrow$ 適切に取引戦略を組めば統計的裁定が可能

理由I: 適切に資産を組み合わせて$V(0) = 0$とできる (仮定)

理由II: $V(T)$が定常 $\Rightarrow$ ${\mathbb E} [V(T)] = \mu_P$となるような時間に依存しない定数$\mu_P$が存在 (うまくやれば$\mu_P \ge 0$にできる)

理由III: $\sigma^2_P = Var(V(T)) = Cov(V(T), V(T))$だから$\sigma^2_P$は時間に依存しない $\Rightarrow$ $\lim_{T \rightarrow \infty} \sigma^2_P / T = 0$

理由IV: ポートフォリオの初期値$V(0) = 0$で${\mathbb E} [V(T)] = \mu_P > 0$

$\Rightarrow$ 取引期間を決めずにポートフォリオの価値が$\mu_P$になるのを待って利益確定すれば良い

#### 共和分過程 (cointegrated process)

(1次の)和分過程 (integrated process) = 単位根過程 (unit-root process): 差分が定常になる確率過程

$M$次元の確率過程${\bf X} (t)$が以下を満足 $\Rightarrow$ 共和分過程

1. ${\bf X} (t)$の各要素が1次の和分過程

2. ${\bf b}^\top {\bf X} (t)$が定常過程となる${\bf b} \ne {\bf 0} \in {\mathbb R}^M$が存在

${\bf b}$: 共和分ベクトル (cointegrated vector)

危険資産の価値ベクトル ${\bf S} (t) = (S_1(t), \ldots, S_M(t))^\top$

VAR(1)モデル

$\Delta {\bf S} (t) = {\bf c} + \Pi {\bf S} (t - 1) + {\bf Z} (t)$

$\Delta {\bf S} (t) = {\bf S} (t) - {\bf S} (t - 1)$

${\bf c}$: const.

$\Pi$: $M \times M$の行列

${\bf Z} (t) = (Z_1 (t), \ldots, Z_M (t))^\top$: ホワイトノイズ

$Z_i \sim N(0, \sigma^2)$

$s = 0$のとき$Cov (Z_i(t), Z_j(t - s)) = \sigma_{i, j}$: 定数

$\Pi \ne {\rm O}$ $\Rightarrow$ ${\bf S} (t)$: 共和分過程

$\Pi = {\rm O}$ $\Rightarrow$ ${\bf S} (t)$: ランダムウォーク (random walk)

In [4]:
!pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 4.8 MB/s eta 0:00:000:00:010:00:01:01

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


##### Johansenの検定

$\Pi = {\rm O}$かどうかを検定

実際には$\Pi = \alpha \beta^\top$として扱う

$\alpha$, $\beta$: $M \times R$行列

In [5]:
# Johansenの検定を試す

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.tsa.api import VECM

# パラメータ設定

Sigma = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])  # Zの共分散行列

a_1 = np.array([0.1, 0.2, 0.3]) # alpha_1
a_2 = np.array([0.5, -0.3, -0.3]) # alpha_2 
b_1 = np.array([1, -0.5, -0.5]) # beta_1
b_2 = np.array([-0.2, 0.5, -0.2]) # beta_2

A = np.stack([a_1, a_2], axis=1) 
B = np.stack([b_1, b_2]) 

Pi_1 = np.dot(a_1.reshape(3, 1), b_1.reshape(1, 3))
Pi_2 = np.dot(A, B)

np.random.seed(seed=1)
rand_nums = np.random.randn(20000, 3)

init = np.array([1, 1, 1])
S_rw = init # random walk
S_ci1 = init # 共和分過程 (rank 1)
S_ci2 = init # 共和分過程 (rank 2)

for rand_num in rand_nums:
    S_rw = np.vstack((S_rw, S_rw[len(S_rw) - 1] + np.dot(rand_num, Sigma)))
    S_ci1 = np.vstack((S_ci1, np.dot(Pi_1 + np.eye(3), S_ci1[len(S_ci1) - 1]) + np.dot(rand_num, Sigma)))
    S_ci2 = np.vstack((S_ci2, np.dot(Pi_2 + np.eye(3), S_ci2[len(S_ci2) - 1])  + np.dot(rand_num, Sigma)))
    
# Johansen検定の実行
# 最初はrandom walkを対象に

JohansenTestResult_rw = coint_johansen(S_rw, k_ar_diff=0, det_order=-1) # 定数項を0に
print(JohansenTestResult_rw.lr1) # trace statistic

# 帰無仮説Piのランク=0に対する統計量，…Piのランク=1…，…Piのランク=2…

[22.58036266  8.78820583  2.43645402]


In [6]:
# 棄却統計量 (90%, 95%, 99%)

print(JohansenTestResult_rw.cvt)

# Piのランク=0の帰無仮説を有意水準10%でのみ何とか棄却

[[21.7781 24.2761 29.5147]
 [10.4741 12.3212 16.364 ]
 [ 2.9762  4.1296  6.9406]]


In [7]:
print(JohansenTestResult_rw.lr2) # maximum eigenvalue statistic

[13.79215683  6.35175181  2.43645402]


In [8]:
# 棄却統計量 (90%, 95%, 99%)

print(JohansenTestResult_rw.cvm)

# 帰無仮説を全く棄却できない

[[15.7175 17.7961 22.2519]
 [ 9.4748 11.2246 15.0923]
 [ 2.9762  4.1296  6.9406]]


In [9]:
# 共和分過程 (rank 1)に対して実行

JohansenTestResult_s1 = coint_johansen(S_ci1, k_ar_diff=0, det_order=-1)
print(JohansenTestResult_s1.lr1) # trace statistic
print(JohansenTestResult_s1.cvt)
print(JohansenTestResult_s1.lr2) # maximum eigenvalue statistic
print(JohansenTestResult_s1.cvm)

# ランク0に対して99%の棄却域を軽く超えている
# 他のランクに対しては帰無仮説が棄却できない

[1.15097289e+04 1.42336157e+01 1.08408520e+00]
[[21.7781 24.2761 29.5147]
 [10.4741 12.3212 16.364 ]
 [ 2.9762  4.1296  6.9406]]
[1.14954952e+04 1.31495305e+01 1.08408520e+00]
[[15.7175 17.7961 22.2519]
 [ 9.4748 11.2246 15.0923]
 [ 2.9762  4.1296  6.9406]]


In [10]:
# 共和分過程 (rank 2)に対して実行

JohansenTestResult_s2 = coint_johansen(S_ci2, k_ar_diff=0, det_order=-1)
print(JohansenTestResult_s2.lr1) # trace statistic
print(JohansenTestResult_s2.cvt)
print(JohansenTestResult_s2.lr2) # maximum eigenvalue statistic
print(JohansenTestResult_s2.cvm)

# ランク2だけは帰無仮説が棄却できない

[3.24178966e+04 6.11019131e+03 2.57568342e+00]
[[21.7781 24.2761 29.5147]
 [10.4741 12.3212 16.364 ]
 [ 2.9762  4.1296  6.9406]]
[2.63077053e+04 6.10761563e+03 2.57568342e+00]
[[15.7175 17.7961 22.2519]
 [ 9.4748 11.2246 15.0923]
 [ 2.9762  4.1296  6.9406]]


In [11]:
# 1次の共和分過程のパラメータ推定

model_s1 = VECM(S_ci1, k_ar_diff=0, coint_rank=1, deterministic='na')
res_s1 = model_s1.fit()
res_s1.summary()

# alpha = (0.1, 0.2, 0.3), beta = (1, -0.5, -0.5)で与えたことに注意

,coef,std err,z,P>|z|,[0.025,0.975]
ec1,0.0998,0.003,32.514,0.000,0.094,0.106
,coef,std err,z,P>|z|,[0.025,0.975]
ec1,0.2077,0.003,68.038,0.000,0.202,0.214
,coef,std err,z,P>|z|,[0.025,0.975]
ec1,0.3019,0.003,98.418,0.000,0.296,0.308
,coef,std err,z,P>|z|,[0.025,0.975]
beta.1,1.0000,0,0,0.000,1.000,1.000
beta.2,-0.5000,0.000,-1849.489,0.000,-0.501,-0.499
beta.3,-0.4997,0.000,-1218.700,0.000,-0.501,-0.499


In [12]:
# 2次の共和分過程のパラメータ推定

model_s2 = VECM(S_ci2, k_ar_diff=0, coint_rank=2, deterministic='na')
res_s2 = model_s2.fit()
res_s2.summary()

# alphaとbetaの誤差が大きいように見える

,coef,std err,z,P>|z|,[0.025,0.975]
ec1,0.0030,0.003,1.024,0.306,-0.003,0.009
ec2,0.1983,0.004,45.903,0.000,0.190,0.207
,coef,std err,z,P>|z|,[0.025,0.975]
ec1,0.2643,0.003,92.164,0.000,0.259,0.270
ec2,-0.2566,0.004,-59.728,0.000,-0.265,-0.248
,coef,std err,z,P>|z|,[0.025,0.975]
ec1,0.3602,0.003,125.012,0.000,0.355,0.366
ec2,-0.2988,0.004,-69.211,0.000,-0.307,-0.290
,coef,std err,z,P>|z|,[0.025,0.975]
beta.1,1.0000,0,0,0.000,1.000,1.000


In [13]:
# Piの形にしてみる

print(np.dot(res_s2.alpha, res_s2.beta.transpose()))

# 元の形は[[0, 0.2, -0.15], [0.26, -0.25, -0.04], [0.36, -0.3, -0.09]]

[[ 0.00295261  0.19832263 -0.15132406]
 [ 0.26430882 -0.25658122 -0.03882909]
 [ 0.36024384 -0.29875471 -0.09113987]]
